# 06 - Commuting modes by commune in Ile-de-France

**Source: MOBPRO 2023 only, scanned in full.** This notebook estimates workers by their
usual main commuting mode: car/van/truck, public transport, bicycle, walking and similar,
motorised two-wheelers, and no transport. It does not describe all travel by all residents,
individual journey legs or daily passenger counts.

No station, line, cycling or other dataset is joined. Mode labels are English translations
of the six TRANS codes already reviewed in the MOBPRO dictionary.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.io as pio
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Open the hackathon project folder.')
if not Path(sys.executable).resolve().is_relative_to(ROOT / '.venv'):
    raise RuntimeError('Select the project .venv kernel, then restart and run all.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.datasets import mobpro

pio.renderers.default = 'plotly_mimetype'
print('Python:', sys.executable)


Python: D:\Hackathon Mobilités 2026 Spécial Dataviz\.venv\Scripts\python.exe


## Choose what the commune represents

- **residence**: workers who live in an IDF commune (`REGION = 11`), including jobs outside IDF.
- **workplace**: workers whose job is in an IDF commune (`REGLT = 11`), including residents outside IDF.

Change GEOGRAPHY and rerun the next cell to switch populations. Paris workplace
arrondissements are combined into commune `75056` for commune-level comparison.
The full file is scanned in chunks and reduced to small aggregates; this may take a short while.
No data is written to disk.


In [2]:
GEOGRAPHY = 'residence'  # Or 'workplace'.
summary = mobpro.commune_mode_summary(ROOT / mobpro.FILENAME, geography=GEOGRAPHY)

print('Population by:', GEOGRAPHY)
print('Communes:', summary.commune_code.nunique())
print('Source records:', f'{summary.sample_records.sum():,}')
print('Estimated workers:', f'{summary.estimated_workers.sum():,.0f}')
assert summary.groupby('commune_code').share_pct.sum().sub(100).abs().max() < 1e-8
display(summary.head(12))


Population by: residence
Communes: 1266
Source records: 1,716,315
Estimated workers: 5,760,838


,commune_code,TRANS,sample_records,estimated_workers,commune_estimated_workers,share_pct,mode_label
0,75056,1,16209,53572.53374,1079451.83791,4.962939,No transport
1,75056,2,38908,125410.480537,1079451.83791,11.617978,Walking and similar
2,75056,3,38599,126093.982345,1079451.83791,11.681298,Bicycle (including electric)
3,75056,4,10107,32841.50884,1079451.83791,3.042425,Motorised two-wheelers
4,75056,5,28529,90723.409003,1079451.83791,8.404581,Car / van / truck
5,75056,6,205697,650809.923445,1079451.83791,60.290779,Public transport
6,77001,1,10,52.70303,625.731855,8.422622,No transport
7,77001,4,2,10.346449,625.731855,1.653496,Motorised two-wheelers
8,77001,5,100,525.880641,625.731855,84.042491,Car / van / truck
9,77001,6,7,36.801736,625.731855,5.881391,Public transport


## Actual geography columns and commune names

In the raw `FD_MOBPRO_2023.csv`:

| Column | Meaning |
|---|---|
| `COMMUNE` | INSEE residence commune code, including its department code |
| `DCLT` | Workplace commune code; Paris, Lyon and Marseille use municipal arrondissements |
| `ARM` | Residence municipal arrondissement for Paris, Lyon and Marseille |
| `DCFLT` | Cross-border workplace commune code (not the main IDF workplace field) |
| `REGION` / `REGLT` | Residence / workplace REGION codes, not commune codes; `11` is IDF |

Our `commune_code` is a derived column: COMMUNE for residence, or DCLT with Paris grouped
into `75056` for workplace. The source records contain codes rather than commune names.

The local variable dictionary stores codes in `COD_MOD`, names in `LIB_MOD`, and the
corresponding variable in `COD_VAR`. Below we use its `COMMUNE` entries for commune names.
This only adds dictionary labels; transport datasets remain separate. Unknown names are
shown explicitly rather than guessed.


In [3]:
from scripts.datasets import mobpro_dictionary

# Use the same statistical dictionary as the source, not postal-code lookups.
dictionary = mobpro_dictionary.load(ROOT / mobpro_dictionary.FILENAME)
commune_names = dictionary.loc[dictionary.COD_VAR.eq('COMMUNE')].set_index('COD_MOD')['LIB_MOD']
summary['commune_name'] = summary.commune_code.map(commune_names).fillna('Unknown commune')
summary['commune_label'] = summary.commune_name + ' (' + summary.commune_code + ')'

display(summary[['commune_code', 'commune_name', 'commune_label']].drop_duplicates().head(15))
print('Commune codes with unknown names:', summary.loc[
    summary.commune_name.eq('Unknown commune'), 'commune_code'
].nunique())


,commune_code,commune_name,commune_label
0,75056,Paris,Paris (75056)
6,77001,Achères-la-Forêt,Achères-la-Forêt (77001)
10,77002,Amillis,Amillis (77002)
14,77003,Amponville,Amponville (77003)
17,77004,Andrezel,Andrezel (77004)
21,77005,Annet-sur-Marne,Annet-sur-Marne (77005)
27,77006,Arbonne-la-Forêt,Arbonne-la-Forêt (77006)
31,77007,Argentières,Argentières (77007)
36,77008,Armentières-en-Brie,Armentières-en-Brie (77008)
40,77009,Arville,Arville (77009)


Commune codes with unknown names: 0


## Familiar workplace commune codes

The cell below lists examples and checks that their workplace codes actually occur in MOBPRO.
These identify communes (or a Paris arrondissement), not individual employers or business parks.
Use `code_for_commune_charts` for `COMMUNE_CODE` or `COMPARE_COMMUNES`.
To study people working there, set `GEOGRAPHY = "workplace"` above and rerun the summary cell.
Raw Paris workplace codes use arrondissements; the charts group them under `75056`.


In [4]:
# Familiar employment destinations: commune codes, not individual business parks.
workplace_examples = pd.DataFrame([
    ('Roissy-en-France', '95527'),
    ('Saint-Denis', '93066'),
    ('Nanterre', '92050'),
    ('Courbevoie', '92026'),
    ('Puteaux', '92062'),
    ('Boulogne-Billancourt', '92012'),
    ('Issy-les-Moulineaux', '92040'),
    ('Guyancourt', '78297'),
    ('Vélizy-Villacoublay', '78640'),
    ('Massy', '91377'),
    ('Orly', '94054'),
    ('Paris 8e arrondissement', '75108'),
], columns=['workplace_commune', 'raw_DCLT_code'])

# Check actual workplace records, independently of the GEOGRAPHY setting above.
observed_workplace_codes = set()
for chunk in mobpro.read_chunks(ROOT / mobpro.FILENAME, columns=['DCLT', 'REGLT']):
    observed_workplace_codes.update(chunk.loc[chunk.REGLT.eq('11'), 'DCLT'].unique())

workplace_examples['present_in_MOBPRO'] = workplace_examples.raw_DCLT_code.isin(observed_workplace_codes)
# The charts combine all Paris workplace arrondissements into commune 75056.
workplace_examples['code_for_commune_charts'] = workplace_examples.raw_DCLT_code.replace({'75108': '75056'})
display(workplace_examples)


,workplace_commune,raw_DCLT_code,present_in_MOBPRO,code_for_commune_charts
0,Roissy-en-France,95527,True,95527
1,Saint-Denis,93066,True,93066
2,Nanterre,92050,True,92050
3,Courbevoie,92026,True,92026
4,Puteaux,92062,True,92062
5,Boulogne-Billancourt,92012,True,92012
6,Issy-les-Moulineaux,92040,True,92040
7,Guyancourt,78297,True,78297
8,Vélizy-Villacoublay,78640,True,78640
9,Massy,91377,True,91377


## Inspect one commune

Use its five-character INSEE code, not its postal code. Examples: Paris `75056`,
Saint-Denis `93066`, Roissy-en-France `95527`.

`sample_records` counts observed records. `estimated_workers` sums the statistical weights
IPONDI. `share_pct` divides each mode's estimated workers by all included workers in that
commune. The denominator includes mode 1 (no transport).


In [5]:
COMMUNE_CODE = '93066'
selected = summary.loc[summary.commune_code.eq(COMMUNE_CODE)]
display(selected[['commune_code', 'commune_name', 'TRANS', 'mode_label', 'sample_records',
                  'estimated_workers', 'share_pct']].round(2))

mobpro.plot_commune_modes(summary, COMMUNE_CODE, geography=GEOGRAPHY).show()
mobpro.plot_commune_modes(summary, COMMUNE_CODE, geography=GEOGRAPHY,
                         metric='estimated_workers').show()


,commune_code,commune_name,TRANS,mode_label,sample_records,estimated_workers,share_pct
4920,93066,Saint-Denis,1,No transport,459,1671.74,2.77
4921,93066,Saint-Denis,2,Walking and similar,1261,4148.64,6.88
4922,93066,Saint-Denis,3,Bicycle (including electric),701,2258.37,3.75
4923,93066,Saint-Denis,4,Motorised two-wheelers,280,951.1,1.58
4924,93066,Saint-Denis,5,Car / van / truck,4189,14509.23,24.06
4925,93066,Saint-Denis,6,Public transport,11641,36754.66,60.96


## Compare a few communes

Shares help compare communes with different numbers of workers. Every stacked bar totals
100%. Counts are estimates of workers, not trips or guaranteed numbers of people who
could change modes. Small source-record counts require caution. The table remains
available alongside the chart as a text alternative.


In [6]:
COMPARE_COMMUNES = ['75056', '93066', '95527']
comparison = summary.loc[summary.commune_code.isin(COMPARE_COMMUNES)]
display(comparison[['commune_code', 'commune_name', 'mode_label', 'sample_records',
                    'estimated_workers', 'share_pct']].round(2))
mobpro.plot_commune_comparison(summary, COMPARE_COMMUNES, geography=GEOGRAPHY).show()


,commune_code,commune_name,mode_label,sample_records,estimated_workers,share_pct
0,75056,Paris,No transport,16209,53572.53,4.96
1,75056,Paris,Walking and similar,38908,125410.48,11.62
2,75056,Paris,Bicycle (including electric),38599,126093.98,11.68
3,75056,Paris,Motorised two-wheelers,10107,32841.51,3.04
4,75056,Paris,Car / van / truck,28529,90723.41,8.4
5,75056,Paris,Public transport,205697,650809.92,60.29
4920,93066,Saint-Denis,No transport,459,1671.74,2.77
4921,93066,Saint-Denis,Walking and similar,1261,4148.64,6.88
4922,93066,Saint-Denis,Bicycle (including electric),701,2258.37,3.75
4923,93066,Saint-Denis,Motorised two-wheelers,280,951.1,1.58


## Interpretation

These charts describe usual main commuting modes of the employed population represented
in MOBPRO. They do not establish why a person uses a car, whether a public-transport
alternative exists, or how much of a multimodal journey is cycled. High car use alone
does not establish car dependence. Source weights must be positive and finite; unknown
mode codes or missing communes stop the calculation for inspection.
